## GSAT trend patterns

In [1]:
# In[1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess

In [2]:
import src.slurm_cluster as scluster
client, scluster = scluster.init_dask_slurm_cluster()

/home/m/m301036/.conda/envs/mykernel/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42067 instead
  warnings.warn(


#!/usr/bin/env bash

#SBATCH -J dask-worker
#SBATCH -e /scratch/m/m301036/dask_logs//dask-worker-%J.err
#SBATCH -o /scratch/m/m301036/dask_logs//dask-worker-%J.out
#SBATCH -p compute
#SBATCH -A mh0033
#SBATCH -n 1
#SBATCH --cpus-per-task=64
#SBATCH --mem=256G
#SBATCH -t 06:00:00

/home/m/m301036/.conda/envs/mykernel/bin/python -m distributed.cli.dask_worker tcp://10.128.10.130:46155 --name dummy-name --nthreads 1 --memory-limit 4.00GiB --nworkers 64 --nanny --death-timeout 60 --local-directory /scratch/m/m301036/dask_temp/ --interface ib0



In [3]:
models = ['CanESM5', 'CESM2', 'IPSL_CM6A', 'EC_Earth3', 'ACCESS', 'MPI_ESM', 'MIROC6']
from pathlib import Path
base_dir = Path("/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3")

ICV_ds = {}  # dict: model -> Dataset

for m in models:
    dir_ICV_input = base_dir / m / "SMILE_internal"
    # adjust filename pattern if needed
    fn = dir_ICV_input / f"{m}_SMILE_noise_trend_std_sliding_1950_2022.nc"
    
    print(f"Opening {fn}")
    ICV_ds[m] = xr.open_dataset(fn)

Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/CanESM5/SMILE_internal/CanESM5_SMILE_noise_trend_std_sliding_1950_2022.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/CESM2/SMILE_internal/CESM2_SMILE_noise_trend_std_sliding_1950_2022.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/IPSL_CM6A/SMILE_internal/IPSL_CM6A_SMILE_noise_trend_std_sliding_1950_2022.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/EC_Earth3/SMILE_internal/EC_Earth3_SMILE_noise_trend_std_sliding_1950_2022.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/ACCESS/SMILE_internal/ACCESS_SMILE_noise_trend_std_sliding_1950_2022.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MPI_ESM/SMILE_internal/MPI_ESM_SMILE_noise_trend_std_sliding_1950_2022.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MIROC6/SMILE_internal/MIROC6_SMILE_noise_trend_std_sliding_1950_2022.nc


In [4]:
ICV_ds["ACCESS"]

<xarray.Dataset>
Dimensions:          (period: 64, lat: 90, lon: 180)
Coordinates:
  * period           (period) object '2013-2022' '2012-2022' ... '1950-2022'
  * lat              (lat) float64 -89.0 -87.0 -85.0 -83.0 ... 85.0 87.0 89.0
  * lon              (lon) float64 0.0 2.0 4.0 6.0 ... 352.0 354.0 356.0 358.0
Data variables:
    noise_trend_std  (period, lat, lon) float64 ...

In [5]:
ICV_ds["CESM2"]

<xarray.Dataset>
Dimensions:          (period: 64, lat: 90, lon: 180)
Coordinates:
  * period           (period) object '2013-2022' '2012-2022' ... '1950-2022'
  * lat              (lat) float64 -89.0 -87.0 -85.0 -83.0 ... 85.0 87.0 89.0
  * lon              (lon) float64 0.0 2.0 4.0 6.0 ... 352.0 354.0 356.0 358.0
Data variables:
    noise_trend_std  (period, lat, lon) float64 ...

In [6]:
# calculate the mean of SMILE std pattern
# MMEM trend can be calculated by averaging the trend from all models
ICV_trend_da = xr.concat([ICV_ds['CanESM5'].noise_trend_std,ICV_ds['IPSL_CM6A'].noise_trend_std,ICV_ds['CESM2'].noise_trend_std,
                         ICV_ds['EC_Earth3'].noise_trend_std,ICV_ds['ACCESS'].noise_trend_std,
                         ICV_ds['MPI_ESM'].noise_trend_std,ICV_ds['MIROC6'].noise_trend_std], dim='model', coords='minimal')
MMEM_ICV_trend_da = ICV_trend_da.mean(dim='model')

In [7]:
ICV_trend_da

<xarray.DataArray 'noise_trend_std' (model: 7, period: 64, lat: 90, lon: 180)>
array([[[[0.73266601, 0.7326665 , 0.73634023, ..., 0.73460324,
          0.74197445, 0.74070726],
         [0.78976253, 0.79420022, 0.79512197, ..., 0.77104425,
          0.77335702, 0.77765137],
         [0.93065953, 0.93550378, 0.94593477, ..., 0.89571454,
          0.90298192, 0.91263936],
         ...,
         [1.50788143, 1.55349522, 1.57077326, ..., 1.39780018,
          1.43052085, 1.45851605],
         [1.42613214, 1.43614432, 1.44263687, ..., 1.39998926,
          1.4222681 , 1.42703004],
         [1.44918625, 1.44914243, 1.44308505, ..., 1.41994614,
          1.42981081, 1.44209547]],

        [[0.65897983, 0.65895633, 0.65327899, ..., 0.65906442,
          0.66485959, 0.65432363],
         [0.68701805, 0.6819593 , 0.68173989, ..., 0.67986778,
          0.68312798, 0.68592571],
         [0.83427392, 0.83972835, 0.82180342, ..., 0.79046546,
          0.79960214, 0.81165067],
...
         [0.07687096, 0.07704306, 0.07743563, ..., 0.07571728,
          0.07635887, 0.07620014],
         [0.07215556, 0.07229884, 0.07262498, ..., 0.07098481,
          0.07139309, 0.07243448],
         [0.06554891, 0.06569433, 0.06577208, ..., 0.06537828,
          0.06544197, 0.06537787]],

        [[0.04821762, 0.04825226, 0.0484117 , ..., 0.04750407,
          0.04777253, 0.04811953],
         [0.04806454, 0.04788377, 0.04773267, ..., 0.04681312,
          0.04736872, 0.04775174],
         [0.04883105, 0.04853302, 0.04872169, ..., 0.04924572,
          0.04882306, 0.0486075 ],
         ...,
         [0.0738303 , 0.07382723, 0.07420023, ..., 0.07320362,
          0.07300664, 0.07391465],
         [0.06922013, 0.0694075 , 0.06971275, ..., 0.06869957,
          0.06896753, 0.06927991],
         [0.06385824, 0.0639111 , 0.06376888, ..., 0.06383372,
          0.06391171, 0.06391074]]]])
Coordinates:
  * period   (period) object '2013-2022' '2012-2022' ... '1951-2022' '1950-2022'
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
Dimensions without coordinates: model
Attributes:
    units:        K/decade
    description:  Std over run of MK trend of internal SAT anomalies for slid...

In [8]:
# output the ensemble mean trend
import os
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MMLE/SMILE_internal/'
os.makedirs(dir_out, exist_ok=True)
MMEM_ICV_trend_da.to_dataset(name='trend').to_netcdf(dir_out + 'MMLE_internal_trend_std_1950-2022_sliding.nc')

In [9]:
client.close()
scluster.close()